# Week 11: ML Review

### Setup

Run the following 2 cells to import all necessary libraries and helpers for this homework

In [ ]:
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/data_utils.py
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/image_utils.py
!wget -qO- https://github.com/PSAM-5020-2026S-A/5020-utils/releases/latest/download/lfw.tar.gz | tar xz

In [ ]:
from random import randrange

from sklearn.metrics import classification_report, confusion_matrix

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score

import pandas as pd

from data_utils import LFWUtils
from data_utils import classification_error, display_confusion_matrix
from image_utils import make_image

## Face Unlock

Let's train a model to detect our face. We can think of this as a simpler version of one of the components inside something like the face ID software on our phones.

We'll skip the face detection part, which is when we find faces in an image, and assume we can get cropped and aligned faces out of images or video streams. We'll look at face detection later in the semester.

This is a slightly different kind of problem from the classification exercise we did in class, but the process is mostly the same.

We will use a dataset with other people's faces, but in the end we are only interested on how well our model detects our face.

### We Always Start with the Data

The dataset we're using is inside `./data/images/lfw/cropped`. It's a subset of the [Labeled Faces in the Wild](https://vis-www.cs.umass.edu/lfw/) dataset.

Take a look at the directory.

What's there?

How's the data organized and labeled?

### Loading the Data

Since we're not interested in generic classification, and measuring how we do on unlabeled data, this whole dataset is labeled, and we can read it into `train` and `test` subsets by calling the `train_test_split()` function of the `LFWUtils` class.

This function takes an optional parameter that specifies what portion of the data should be used for the `test` dataset. We can start with the default value of $0.5$.

In [ ]:
train, test = LFWUtils.train_test_split(dir="./data/image/lfw/cropped", test_pct=0.5)

### Looking at the Data

Ok. Data is loaded.

What's in the data? How is it actually organized?

Take a look at the objects that were returned in each of the $2$ variables.

How big are our datasets?

Take a look at the `LABELS` and `L2I` members of the `LFWUtils` class (`LFWUtils.LABELS` and `LFWUtils.L2I`).

In [ ]:
# TODO: look at dataset objects (train and test variables). What's in them?

# TODO: how big are them? (how many records?)
print(f"training records: {len(train)}")
print(f"testing records: {len(test)}")

# TODO: how many labels do they have?
# TODO: what are the labels

print(f"number of labels: {len(LFWUtils.LABELS)}")
print(f"labels : {LFWUtils.LABELS}")

print(f"training samples: {len(train['labels'])}")
print(f"training labels: {train['labels'][:20]}")

print(f"test samples: {len(test['labels'])}")
print(f"test labels: {test['labels'][:20]}")

### Visualizing the Data

We can open some random images to make sure the content of our datasets make sense.

Our `LFWUtils` class has some member variables that hold the image dimensions (`LFWUtils.IMAGE_SIZE`, `LFWUtils.IMAGE_WIDTH`, `LFWUtils.IMAGE_HEIGHT`)

In [ ]:
train_size = len(train["labels"])
ridx = randrange(0, train_size)

label_id = train["labels"][ridx]

display(make_image(train["pixels"][ridx], width=LFWUtils.IMAGE_SIZE[0]))

print("id:", label_id,
      "\nlabel:", LFWUtils.LABELS[label_id],
      "\nfrom:", train["files"][ridx])

### Adding your images

Create a directory in the `dataset` directory for your images. Give it a one-word name, like your last name, your New School id or your initials. For example, mine is called `tgh` and is located at: `./data/images/lfw/cropped/tgh`.

Now, add between $20$ and $30$ images of your face to your directory. 

The images should be just like the ones that are already there for the other people:
- $130$ pixels wide
- $170$ pixels tall
- single-channel grayscale
- jpeg format
- named `label-number.jpg` (for example: `tgh-000.jpg`)

Feel free to do this manually using Photoshop or any other image editing software, but the easiest way is to use this interface that automatically crops faces out of pictures and creates images in the correct format:

### [Face Align](https://huggingface.co/spaces/visualizedata/PSAM5020-FaceAlign-Gradio)

It will also align the faces and put the eyes in a consistent location. There's even an option to capture pictures from a live camera stream.

### Reload Dataset

Just run the `train_test_split()` again.

### PCA, Classification, etc etc etc

Now that we have added our images to the dataset, let's train a classifier and see how well it performs on not just classification, but on recognizing our face.

The images are $130$ x $170$ ($22\text{,}100$ pixels), so let's do `PCA`. We can aim for an explained variance value of about $80\%$, and adjust that later if we find necessary.

Once we have the PCs for our training dataset in a `DataFrame` we can add a `label` column to it with the correct labels we have in `train["labels"]`.

We can also create a `DataFrame` for testing now by using the same `PCA` object to `transform()` the `test["pixels"]` data.

Since we won't train anything with the test dataset, it's ok to just keep the labels in `test["labels"]` as they are.

In [ ]:
# TODO: create PCA, fit and transform train data
face_pca = PCA(n_components=0.95)
x_train_pca = face_pca.fit_transform(train["pixels"])

# check if pca actually changed based on number of features being use 
print(f"number of features being used: {face_pca.n_components_}")

# TODO: check PCA captured variance
captured_variance = face_pca.explained_variance_ratio_.sum()
print(captured_variance) 

# TODO: prepare DataFrame for training (add label column)
train_df = pd.DataFrame(x_train_pca)
train_df["label"] = train["labels"]

# TODO: create the test DataFrame by running PCA on the test data
x_test_pca = face_pca.transform(test["pixels"])
df_test = pd.DataFrame(x_test_pca)

display(train_df.head())

We can use the following cell to take a look at our images and their reconstructions.

This assumes the `DataFrame` is called `train_df` and the `PCA` object is called `face_pca`. Adjust these if necessary.

In [ ]:
train_size = len(train["labels"])
ridx = randrange(0, train_size)

# reconstruct image
pca_pixels = face_pca.inverse_transform(train_df.iloc[[ridx]].drop(columns=["label"]))

display(make_image(train["pixels"][ridx], width=LFWUtils.IMAGE_SIZE[0]))
display(make_image(pca_pixels, width=LFWUtils.IMAGE_SIZE[0]))

In [ ]:
# filter the DataFrame by our label
awesome_df = train_df[train_df["label"] == LFWUtils.L2I["lula"]]

# index of image with our label
img_idx = 0
awesome_idx = awesome_df.index[img_idx]

# reconstruct image
pca_pixels = face_pca.inverse_transform(awesome_df.iloc[[img_idx]].drop(columns=["label"]))

display(make_image(train["pixels"][awesome_idx], width=LFWUtils.IMAGE_WIDTH))
display(make_image(pca_pixels, width=LFWUtils.IMAGE_WIDTH))

### Interpretation

<span style="color:hotpink;">
Do these make sense ? Do they look "recognizable" ? How do they change as a function of `PCA` <code>n_components</code> ?
</span>

<span style="color:hotpink;">EDIT THIS CELL WITH ANSWER</span>

These definitely make sense. The faces and their most important features are still there. I would not necessarily that from the generated images I would have recognized them right away, but we found a good middle ground with the 80% explained variance. 

Now, back to classifying...

Maybe start with `RandomForestClassifier()` or `SVC()`...

In [ ]:
# TODO: create a classifier
classifier = SVC(kernel="linear")

# TODO: separate input and output columns from the train DataFrame
x_train = train_df.drop(columns=["label"])
y_train = train_df["label"]

# TODO: train model using train data and labels
classifier.fit(x_train, y_train)

# TODO: run prediction on train data
train_predictions = classifier.predict(x_train)

### Validate model with training data

In [ ]:
# measure classification error
print("error:", classification_error(train["labels"], train_predictions))

# look at precision/recall from classification_report
print(classification_report(train["labels"], train_predictions))

# look at confusion matrix
display_confusion_matrix(train["labels"], train_predictions, LFWUtils.LABELS)

### Interpretation

<span style="color:hotpink;">
How does the confusion matrix look ? What does it mean ?
</span>

<span style="color:hotpink;">EDIT THIS CELL WITH ANSWER</span>

The confusion matrix looks perfect, which is almost too perfect. It could be an indicator for overfitting. If the test data performs poorly now, it is definitely overfitting. 

### Validate model with testing data

In [ ]:
x_test_pca = face_pca.transform(test["pixels"])
df_test = pd.DataFrame(x_test_pca)
df_test["label"] = test["labels"]

# TODO: run prediction on test data
x_test = df_test.drop(columns=["label"])
y_test = df_test["label"]

test_predictions = classifier.predict(x_test)
error = classification_error(y_test, test_predictions)
print(f"classification Error: {error}")

# TODO: measure classification error
# TODO: look at precision/recall from classification_report
# TODO: look at confusion matrix
print("\nclassification report:")
print(classification_report(y_test, test_predictions, target_names=LFWUtils.LABELS))

# TODO: look at confusion matrix
# Using your imported helper function to visualize the results
display_confusion_matrix(y_test, test_predictions, display_labels=LFWUtils.LABELS)

### Different Classifiers

Try changing the classifier type above, or some of its parameters, to see if the overall accuracy can be improved.

Some things to try:
- `SVC(kernel="linear")`: this changes the type of curve the model tries to use to separate our classes. The more complex default type (`rbf`) might be over-fitting the data. We can experiment with `linear` or `poly` curves.
- `LogisticRegression()`: the classification that fits statistical modeling functions (bell curves) to our data.


### Hyper-parameter Optimization

Each model type has a handful of parameters that can be adjusted and optimized.

For example, we could add "regularization" to a `LogisticRegression` classifier with `C=0.1`.

Regularization is a process that makes training harder by setting more restrictions on the kind of answers it can give.

This can be restrictions like:
- keep all calculated parameters close to $0$
- minimize the number of non-zero parameters used
- keep all parameters positive

The result is slightly worse performance on the training dataset, but hopefully better generalization of the model and better performance on the test dataset.

In [ ]:
# TODO: try LogisticRegression() and/or SVC(kernel="linear")
svc_classifier = SVC(kernel="linear", C=0.5, class_weight='balanced')

svc_classifier.fit(x_train, y_train)
test_predictions = svc_classifier.predict(x_test)

print(f"test error: {classification_error(y_test, test_predictions)}")

print("\nclassification report:")
print(classification_report(y_test, test_predictions, target_names=LFWUtils.LABELS))

# TODO: does adding features (increasing PCA components) help ?
# TODO: look at confusion matrix
display_confusion_matrix(y_test, test_predictions, display_labels=LFWUtils.LABELS)

### Interpretation

<span style="color:hotpink;">
How does THIS confusion matrix look ? What does it mean ? How does it perform for your pictures ?
</span>

<span style="color:hotpink;">EDIT THIS CELL WITH ANSWER</span>

I changed the variance of the pca to 0.95 and used a svc classifier but the confusion matrix is exactly the same for each case. As a conclusion, changing features did not help, 0.80 variance contains enough information for the classifier. 

The model performance is very different for different faces, e.g. koizumi or bushl. Probably because they have more distinct features. 

### Precision and Recall

Accuracy, which is the complement of our `classification_error` value, is the measurement that is optimized during the `RandomForestClassifier` training process.

If we were training a regular classifier, we would look at `accuracy` (or `classification_error`) to determine if our model's performance is acceptable.

Since we're working on a personal face recognition model, we don't really care about overall accuracy, but instead are more interested in the `precision` and `recall` values for the classification of our particular images.

We don't want overall accuracy to be horrible, but we can be more specific in this case and be happy if the correct portion of our confusion matrix looks good.

Calculate the `precision` and `recall` values for the classification of your images.

In [ ]:
# TODO: calculate precision
precision = precision_score(y_test, test_predictions, average='macro')
print(f"precision: {precision}")

# TODO: calculate recall
recall = recall_score(y_test, test_predictions, average='macro')
print(f"recall: {recall}")

### Interpretation

<span style="color:hotpink;">
How is it performing for your images ? Which value, precision or recall, is higher ? What does that mean ?
</span>

<span style="color:hotpink;">EDIT THIS CELL WITH ANSWER</span>

Precision is slightly higher than recall. This indicates the model is better at making predictions than finding all instances of a person. However, both numbers are relatively small and therefor still guesses most of the time.

We can run the following cell to see which classes have the highest `precision` and `recall` scores:

In [ ]:
print("top precision:", LFWUtils.top_precision(test["labels"], test_predictions, top=5))
print("top recall:", LFWUtils.top_recall(test["labels"], test_predictions, top=5))